## Degredation Experiments

In [1]:
from ultralytics import YOLO
fp32_model = YOLO("yolov8n.pt")
print("FP32 Loaded")

fp16_model = YOLO("yolov8n_openvino_model/")
print("FP16 Loaded")

int8_model = YOLO("yolov8n_int8_openvino_model/")
print("INT8 Loaded")

print()
print("ALL Three Models Ready")


FP32 Loaded
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
FP16 Loaded
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
INT8 Loaded

ALL Three Models Ready


In [14]:
import cv2
import time
import pandas as pd
import os


def run_inference(model, frames_folder, model_name, condition_name):
    
    results_list = []
    total_time = 0
    frame_files = sorted(os.listdir(frames_folder))
    frame_count = 0

    for filename in frame_files:
        img_path = os.path.join(frames_folder, filename)
        frame = cv2.imread(img_path)
        if frame is None:
            continue

        start_time = time.time()
        results = model(frame, imgsz=640, conf = 0.25, verbose=False)
       

        end_time = time.time()
        inference_time = end_time - start_time
        total_time += inference_time

        for r in results:
            for box in r.boxes:
                results_list.append({
                    "frame":frame_count,
                    "model":model_name,
                    "condition":condition_name,
                    "confidence":float(box.conf[0]),
                    "class_id":int(box.cls[0]),
                    "inference_time":inference_time
                })
        frame_count += 1
        if frame_count % 50 == 0:
            print(f" Processing frame {frame_count}/367")
            
    df = pd.DataFrame(results_list)
    avg_time = total_time / frame_count
    fps = round(1/avg_time, 1)

    if len(df)>0:
        mean_conf = round(df["confidence"].mean(),3)
        std_conf = round(df["confidence"].std(), 3)
        detections = len(df)
    else:
        mean_conf = 0
        std_conf = 0
        detections = 0

    print(f"{model_name:<6} | {condition_name:<15} | detections={detections:<5} | FPS={fps:<5} | mean_conf={mean_conf}")
    return df

In [15]:
import os
print(os.getcwd())
print(os.listdir("degraded"))

conditions = [
("blur_sigma1",   "degraded/blur_sigma1"),
("blur_sigma2",   "degraded/blur_sigma2"),
("blur_sigma3",   "degraded/blur_sigma3"),
("jpeg_q90",     "degraded/jpeg_q90"),
("jpeg_q70",     "degraded/jpeg_q70"),
("jpeg_q50",     "degraded/jpeg_q50"),
("gaussian_noise", "degraded/gaussian_noise"),
]

all_results = []

print("Starting degradation experiments...")
print("This will take 20-40 minutes on my laptop")
print()

for condition_name, folder_path in conditions:
    df = run_inference(fp32_model, folder_path, "FP32", condition_name)
    all_results.append(df)

print()

for condition_name, folder_path in conditions:
    df = run_inference(fp16_model, folder_path, "FP16", condition_name)
    all_results.append(df)

print()

for condition_name, folder_path in conditions:
    df = run_inference(int8_model, folder_path, "INT8", condition_name)
    all_results.append(df)
    
print()
print("All 21 degraded experiments complete")

fp32_clean = pd.read_csv("outputs/day03_fp32_results.csv")
fp32_clean["model"] ="FP32"
fp32_clean["condition"] = "clean"

fp16_clean = pd.read_csv("outputs/fp16_video_results.csv")
fp16_clean["model"] ="FP16"
fp16_clean["condition"] = "clean"

int8_clean = pd.read_csv("int8_results.csv")
int8_clean["model"] ="INT8"
int8_clean["condition"] = "clean"

all_results.append(fp32_clean)
all_results.append(fp16_clean)
all_results.append(int8_clean)

print("CLean video results added from existing CSVs")
print()

combined_df = pd.concat(all_results, ignore_index = True)

print("TOtal rows in combined results:", len(combined_df))
print("This covers 3 models x 8 conditions = 24 experiments")




C:\Users\faiza\Documents\edge-perception-project
['blur_sigma1', 'blur_sigma2', 'blur_sigma3', 'gaussian_noise', 'jpeg_q50', 'jpeg_q70', 'jpeg_q90']
Starting degradation experiments...
This will take 20-40 minutes on my laptop

 Processing frame 50/367
 Processing frame 100/367
 Processing frame 150/367
 Processing frame 200/367
 Processing frame 250/367
 Processing frame 300/367
 Processing frame 350/367
FP32   | blur_sigma1     | detections=3284  | FPS=9.1   | mean_conf=0.566
 Processing frame 50/367
 Processing frame 100/367
 Processing frame 150/367
 Processing frame 200/367
 Processing frame 250/367
 Processing frame 300/367
 Processing frame 350/367
FP32   | blur_sigma2     | detections=2948  | FPS=7.1   | mean_conf=0.555
 Processing frame 50/367
 Processing frame 100/367
 Processing frame 150/367
 Processing frame 200/367
 Processing frame 250/367
 Processing frame 300/367
 Processing frame 350/367
FP32   | blur_sigma3     | detections=2342  | FPS=7.9   | mean_conf=0.536
 Proces

In [16]:
# DEGRADATION EXPERIMENT FINDINGS
#
# 1. BLUR: Detections drop consistently as sigma increases across all models.
#    FP32, FP16 and INT8 degrade at similar rates — quantisation does not
#    increase vulnerability to blur degradation.
#
# 2. JPEG COMPRESSION: Minimal effect on detections or confidence across
#    all quality levels (Q=90, 70, 50). All three models are robust to
#    JPEG compression artefacts.
#
# 3. GAUSSIAN NOISE: Largest drop in detections and confidence across all
#    models. Noise is the most damaging degradation type tested.
#
# 4. CONFIDENCE CONVERGENCE: Clean video showed a large confidence gap
#    FP32(0.318) vs FP16(0.571) vs INT8(0.700). Under degradation all
#    three models converge to similar confidence levels (0.51-0.57).
#    This suggests the overconfidence seen in quantised models on clean
#    video is partly suppressed by real-world degradation conditions.
#
# 5. SPEED: FP16 and INT8 maintain their speed advantage under degradation.
#    FP16=26-27 FPS, INT8=31-32 FPS, FP32=9-11 FPS on degraded image files.
#    Note: FPS on image files is higher than video capture due to I/O differences.

In [17]:
import pandas as pd
import os

# Save the full combined results to CSV
os.makedirs("outputs", exist_ok=True)
combined_df.to_csv("outputs/degradation_results.csv", index=False)
print("Full results saved to: outputs/degradation_results.csv")
print("Total rows:", len(combined_df))
print()

# Build summary table
summary_rows = []

for model in ["FP32", "FP16", "INT8"]:
    for condition in ["clean", "blur_sigma1", "blur_sigma2", "blur_sigma3",
                      "jpeg_q90", "jpeg_q70", "jpeg_q50", "gaussian_noise"]:
        
        subset = combined_df[(combined_df["model"] == model) & 
                             (combined_df["condition"] == condition)]
        
        if len(subset) == 0:
            continue

        detections = len(subset)
        mean_conf = round(subset["confidence"].mean(), 3)
        std_conf = round(subset["confidence"].std(), 3)
        avg_time = subset["inference_time"].mean()
        fps = round(1 / avg_time, 1)

        summary_rows.append({
            "model": model,
            "condition": condition,
            "detections": detections,
            "mean_conf": mean_conf,
            "std_conf": std_conf,
            "fps": fps
        })

summary = pd.DataFrame(summary_rows)

combined_df.to_csv("outputs/degradation_results.csv", index=False)
summary.to_csv("outputs/degradation_summary.csv", index=False)
print("Saved: outputs/degradation_results.csv")
print("Saved: outputs/degradation_summary.csv")
print()

print("=" * 70)
print(f"{'DEGRADATION EXPERIMENT SUMMARY':^70}")
print("=" * 70)
print(f"{'Model':<8} {'Condition':<18} {'Detections':<13} {'Mean Conf':<12} {'FPS'}")
print("-" * 70)

for _, row in summary.iterrows():
    if row["condition"] == "clean" and row["model"] != "FP32":
        print()
    print(f"{row['model']:<8} {row['condition']:<18} {int(row['detections']):<13} {row['mean_conf']:<12} {row['fps']}")

print("=" * 70)

Full results saved to: outputs/degradation_results.csv
Total rows: 69714

Saved: outputs/degradation_results.csv
Saved: outputs/degradation_summary.csv

                    DEGRADATION EXPERIMENT SUMMARY                    
Model    Condition          Detections    Mean Conf    FPS
----------------------------------------------------------------------
FP32     clean              367           0.318        5.5
FP32     blur_sigma1        3284          0.566        9.0
FP32     blur_sigma2        2948          0.555        6.8
FP32     blur_sigma3        2342          0.536        8.0
FP32     jpeg_q90           3379          0.565        8.7
FP32     jpeg_q70           3339          0.564        8.7
FP32     jpeg_q50           3248          0.558        8.6
FP32     gaussian_noise     2792          0.514        9.3

FP16     clean              3404          0.571        10.1
FP16     blur_sigma1        3294          0.569        24.7
FP16     blur_sigma2        2940          0.557      